# [2] LLM과 친해지기

#1. Hugging Face

##1-1. 필수 라이브러리 설치

In [ ]:
import transformers
print(f"Transformers 버전: {transformers.__version__}")

In [ ]:
!pip install -U "transformers<5"

In [ ]:
import transformers
print(f"Transformers 버전: {transformers.__version__}")

In [ ]:
from transformers import pipeline

##1-2. text-classification

In [ ]:
pipe = pipeline(
    task = "text-classification",
    model = "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

raw_text = [
    "I love you",
    "I hate you",
    "I will meet with you"
]

prediction = pipe(raw_text)
print(prediction)

##1-3. text-generation

In [ ]:
# GPT 모델을 사용한 텍스트 생성
generator = pipeline(
    task = "text-generation",
    model = "gpt2"
)

prompt = "Artificial intelligence will"

result = generator(prompt, truncation = True, max_length = 100, max_new_tokens = None) #default max_new_tokens=256

result

##1-4. question-answering

In [ ]:
from transformers import pipeline

question_answerer = pipeline(task="question-answering",
                             model='distilbert-base-cased-distilled-squad')

context = """
HuggingFace is a company that develops tools for building applications
using machine learning. It is most notable for its transformers library
built for natural language processing applications and its platform that
allows users to share machine learning models and datasets.
"""
qs = [
    "what dose Huggingface develope?",
    "what can users do with Huggingface?"
]

for q in qs:
    result = question_answerer(context = context, question = q)
    print("Q : " + q)
    print("A : " + result["answer"])

##1-5. sentiment-analysis

In [ ]:
generator = pipeline(
    task="sentiment-analysis",
    model="matthewburke/korean_sentiment"
)

texts = [
    "이 영화 정말 재미있어요",
    "실망스러워요"
]

result = generator(texts)

result

#2. Google GenAI

##2-1. 환경변수(API Key 정보) 가져오기

API Key 발급 -> /content/.env 파일 설정

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다:", os.environ["GOOGLE_API_KEY"][:5]+"*****")

구글 라이브러리 설치

In [ ]:
!pip install -q google-genai

##2-2. 모델 호출 - Gemini 2.5 flash

In [ ]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model = "gemini-2.5-flash",
    contents = "AI의 동작 방식을 짧게 설명해줘.",
)

print(response.text)

In [ ]:
from google.genai import types

response = client.models.generate_content(
    model = "gemini-2.5-flash",
    config = types.GenerateContentConfig(
        system_instruction = "너는 고양이야. 네 이름은 네로야."),
    contents = "안녕?"
)

print(response.text)

In [ ]:
response = client.models.generate_content(
    model = "gemini-2.5-flash",
    contents = ["AI의 동작 방식을 짧게 설명해줘."],
    config = types.GenerateContentConfig(
        temperature=0.1
    )
)
print(response.text)

##2-2. 대화하기

💡 Gemini SDK

*   client.chats.create()를 통해 chat이라는 대화 세션 객체 생성
*   chat 객체는 내부적으로 사용자와 AI 간 주고받은 모든 메시지 리스트(대화 기록, History)를 자동으로 누적하여 보관
*   chat.send_message()를 호출할 때마다, 지금까지 저장되어 있던 전체 대화 내역 + 새로운 메시지를 묶어서 AI 모델에 전달

In [ ]:
chat = client.chats.create(model = "gemini-2.5-flash")

message = "강아지 두 마리를 기르고 있어."
response = chat.send_message(message)
print("사용자: " + message)
print("AI 응답: " + response.text)

In [ ]:
message = "내가 강아지 몇 마리 키우지?"
response = chat.send_message(message)
print("사용자: " + message)
print("AI 응답: " + response.text)

In [ ]:
for message in chat.get_history():
    print(f'role - {message.role}', end=": ")
    print(message.parts[0].text)

##2-4. streaming

In [ ]:
from google import genai

client = genai.Client()
chat = client.chats.create(model = "gemini-2.5-flash")

message = "강아지 두마리를 처음 키우게 되었는데, 강아지를 기르는데 주의해야 할 게 뭐가 있을까?"
response = chat.send_message_stream(message)
for chunk in response:
    print(chunk.text, end="")

In [ ]:
message = "내가 강아지 몇 마리 키우지?"
response = chat.send_message_stream(message)
for chunk in response:
    print(chunk.text, end="")

Chat history 확인

In [ ]:
for message in chat.get_history():
    print(message)

#3. Open AI

##3-1. 환경변수(API Key정보) 가져오기

API Key 발급 -> /content/.env 파일 설정

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path = "/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")

OpenAI 라이브러리 설치

In [ ]:
!pip install openai

##3-2. 모델 호출 - gpt 4o mini

In [ ]:
from openai import OpenAI
client = OpenAI()
my_messages = [{"role":"user" , "content": "너는 누구니?"}]
response = client.chat.completions.create(model = "gpt-4o-mini", messages = my_messages)

In [ ]:
answer = response.choices[0].message.content
print(answer)

In [ ]:
response

##3-3. 역할 부여

In [ ]:
my_messages = [
    {"role":"system" , "content": "너는 영어 개인교사 쥴리엣이야"},
    {"role":"user" , "content": "너는 누구니?"}
]
response = client.chat.completions.create(model = "gpt-4o-mini", messages = my_messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
response

##3-4. 대화하기 - 기억X

In [ ]:
my_messages = [
    {"role":"system" , "content": "너는 영어 개인교사 쥴리엣이야"},
    {"role":"user" , "content": "내 이름은 길동이야"}
]
response = client.chat.completions.create(model = "gpt-4o-mini", messages = my_messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
my_messages = [
    {"role":"system" , "content": "너는 영어 개인교사 쥴리엣이야"},
    {"role":"user" , "content": "내 이름은 기억해?"}
]
response = client.chat.completions.create(model = "gpt-4o-mini", messages = my_messages)
answer = response.choices[0].message.content
print(answer)

##3-5. 대화하기 - 기억O

In [ ]:
my_messages = [
    {"role":"system" , "content": "너는 영어 개인교사 쥴리엣이야"},
    {"role":"user" , "content": "내 이름은 길동이야"}
]
response = client.chat.completions.create(model = "gpt-4o-mini", messages = my_messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
my_messages = [
    {"role":"system", "content": "너는 영어 개인교사 쥴리엣이야"},
    {"role":"user", "content": "내 이름은 길동이야"},
    {"role":"assistant", "content": answer},
    {"role":"user", "content": "내 이름은 기억해?"}
]
response = client.chat.completions.create(model = "gpt-4o-mini", messages = my_messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
response

##3-6. TTS

In [ ]:
response = client.audio.speech.create(
    model = "tts-1",
    input = """칸트는 '인간을 결코 도구로 다루지 말라'고 했습니다. 인간을 어떤 목적을 위한 '수단'이 아닌 '목적 그 자체'로 대해야 한다고 강조했습니다.
               AI 시대에서 효율성과 생산성 극대화라는 명목으로 인간을 데이터 수집 대상이나 노동 시장의 단순 도구로 취급해서는 안 됩니다.
               기술이 인간의 행복을 보조하는 수단이 되어야지, 인간이 알고리즘의 통제를 받는 주객전도가 일어나지 않도록 제어해야 합니다.
               AI 시대에 기술의 속도를 다투는 것은 의미가 없습니다. 기계가 '답을 내는 계산기' 역할을 맡아준 덕분에,
               인간은 비로소 '어떻게 다 함께 존엄하고 행복하게 살 것인가'를 사색하는 '철학자'의 삶으로 돌아갈 기회를 얻었습니다.
               기술은 인간을 돕는 도구로 남겨두고, 우리는 공감·책임·사색이라는 인간 고유의 빛을 밝히는 것이 현인들이 제시하는 길입니다.""",
    voice = "nova"
)
with open("test.mp3", "wb") as f:
    f.write(response.content)

In [ ]:
from IPython.display import Audio

Audio("test.mp3")

##3-7. 멀티턴 대화하기

In [ ]:
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")  # 환경 변수에서 API 키 가져오기

client = OpenAI(api_key=api_key)  # 오픈AI 클라이언트의 인스턴스 생성

# 대화 기억 X
while True:
    user_input = input("사용자: ")

    if user_input == "exit":
        break

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0.9,
        messages=[
            {"role": "system", "content": "너는 사용자를 도와주는 상담사야."},
            {"role": "user", "content": user_input},
        ],
    )
    print("AI: " + response.choices[0].message.content)


In [ ]:
from openai import OpenAI  # 오픈AI 라이브러리를 가져오기

api_key = os.getenv("OPENAI_API_KEY")  # 환경 변수에서 API 키 가져오기

client = OpenAI(api_key=api_key)  # 오픈AI 클라이언트의 인스턴스 생성

# 대화 기억 O
def get_ai_response(messages):
    response = client.chat.completions.create(
        model="gpt-4o",  # 응답 생성에 사용할 모델 지정
        temperature=0.9,  # 응답 생성에 사용할 temperature 설정
        messages=messages,  # 대화 기록을 입력으로 전달
    )
    return response.choices[0].message.content  # 생성된 응답의 내용 반환

messages = [
    {"role": "system", "content": "너는 사용자를 도와주는 상담사야."},  # 초기 시스템 메시지
]

while True:
    user_input = input("사용자: ")  # 사용자 입력 받기

    if user_input == "exit":  # 사용자가 대화를 종료하려는지 확인
        break

    messages.append({"role": "user", "content": user_input})  # 사용자 메시지를 대화 기록에 추가
    ai_response = get_ai_response(messages)  # 대화 기록을 기반으로 AI 응답 가져오기
    messages.append({"role": "assistant", "content": ai_response})  # AI 응답 대화 기록에 추가하기

    print("AI: " + ai_response)  # AI 응답 출력


##3-8. 요약하기


*   /test 폴더 생성
*   /test_data.txt 파일 생성




In [ ]:
from openai import OpenAI

api_key = os.getenv('OPENAI_API_KEY')

def summarize_txt(file_path: str): # ① file_path를 매개변수로 받음
    client = OpenAI(api_key=api_key)

    # ② 주어진 텍스트 파일을 읽어들인다.
    with open(file_path, 'r', encoding='utf-8') as f:
        txt = f.read()

    # ③ 요약을 위한 시스템 프롬프트를 생성한다.
    system_prompt = f'''
    너는 다음 글을 요약하는 봇이야. 아래 글을 읽고, 저자의 문제 인식과 주장을 파악하고, 주요 내용을 요약해.
    작성해야 하는 포맷은 다음과 같아.

    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개

    =============== 이하 텍스트 ===============
    { txt }
    '''

    print(system_prompt)
    print('=========================================')

    # ④ OpenAI API를 사용하여 요약을 생성한다.
    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0.1,
        messages=[
            {"role": "system", "content": system_prompt},
        ]
    )

    return response.choices[0].message.content

if __name__ == '__main__':
    file_path = './test/test_data.txt'

    summary = summarize_txt(file_path)
    print(summary)

    # ⑤ 요약된 내용을 파일로 저장한다.
    with open('./test/crop_model_summary.txt', 'w', encoding='utf-8') as f:
        f.write(summary)


###